# 🎯 뉴스 분류 파이프라인 실습 (Student)

- **데이터 소개**
  - 이 실습에서는 **KLUE의 YNAT 뉴스 주제 분류 데이터**를 사용합니다.
  - 이 데이터는 **연합뉴스 뉴스 제목**을 바탕으로 만든 한국어 분류 데이터이며, 기사 본문이 아니라 **제목만 보고** 주제를 예측합니다.
  - 분류 범주는 **정치, 경제, 사회, 생활문화, 세계, IT/과학, 스포츠**의 7개입니다.
  - 데이터와 벤치마크 정보는 `https://klue-benchmark.com/`, 데이터셋은 `https://huggingface.co/datasets/klue/klue`에서 확인할 수 있습니다.
- **실습 목표**
  - 이번 실습에서는 이 데이터를 **TF-IDF + RF/LogReg**으로 분류하고, 성적표와 혼동행렬을 해석합니다.

> **🖥️ 환경 설정**: `uv run jupyter lab`으로 실행하세요.
> 추가 패키지 필요 시: `uv add koreanize-matplotlib scikit-learn seaborn`

> **⌨️ 단축키 안내**
> | 단축키 | 동작 |
> |--------|------|
> | `Shift + Enter` | 셀 실행 후 다음 셀 이동 |
> | `Esc → A` | 위에 셀 삽입 |
> | `Esc → B` | 아래에 셀 삽입 |
> | `Esc → DD` | 셀 삭제 |

In [1]:
!uv pip install koreanize-matplotlib scikit-learn seaborn -q

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

%config InlineBackend.figure_format = 'retina'

## 📂 데이터 로드

KLUE 뉴스 데이터를 로드하고 구조를 확인합니다. 이론 수업에서 배운 파이프라인의 첫 단계입니다.

[→ §100 강의노트](../notes/UD-06-100__text-classification-intro.md)

In [ ]:
# 📌 §100 데이터 로드
# 코드를 작성해주세요 👇

# 예상 출력: 학습: (45654, 3), 시험: (9131, 2)

In [ ]:
# 📌 §100 데이터 확인
# 코드를 작성해주세요 👇

# 예상 출력: index, title, topic_idx 열이 보이는 DataFrame

In [ ]:
# 📌 §100 카테고리 확인
# 코드를 작성해주세요 👇

# 예상 출력: 7개 카테고리 (0:IT과학 ~ 6:정치), 4824~7629건

## 🧹 전처리

숫자를 제거하고 영문을 소문자로 바꿉니다. 형태소 분석 없이 TF-IDF ngram만으로도 충분한 성능이 나옵니다.

[→ §400 강의노트](../notes/UD-06-400__demo-classification-report.md)

In [ ]:
# 📌 §400 전처리
# 코드를 작성해주세요 👇

# 예상 출력: 숫자가 제거된 뉴스 제목 3건

## 🔢 TF-IDF 벡터화

W05에서 배운 TF-IDF를 적용합니다. 학습 데이터로 단어 사전을 만들고(fit), 학습/시험 데이터를 모두 벡터로 변환합니다(transform).

[→ §400 강의노트](../notes/UD-06-400__demo-classification-report.md)

[🔗 TfidfVectorizer 공식 문서](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
# 📌 §400 TF-IDF 생성
# 코드를 작성해주세요 👇

# 예상 출력: 단어 사전 크기: 22385

In [ ]:
# 📌 §400 벡터 변환
# 코드를 작성해주세요 👇

# 예상 출력: X_train: (45654, 22385), X_test: (9131, 22385)

## 🌲 RandomForest 학습

100개 트리가 투표하는 랜덤 포레스트로 뉴스를 분류합니다. 교차검증으로 정확도를 확인합니다.

[→ §300 강의노트](../notes/UD-06-300__rf-vs-logreg.md)

[🔗 RandomForestClassifier 공식 문서](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

> ⏱️ **실행 시간 안내**: 아래 셀은 3-fold 교차검증으로 RF 100개 트리를 학습합니다. **1~3분** 소요됩니다. 실행 후 기다려주세요.

### cross_val_predict가 어떻게 예측을 모으는가

```text
데이터를 3등분
  Fold 1 | Fold 2 | Fold 3

1차
  학습: Fold 2 + Fold 3
  예측: Fold 1

2차
  학습: Fold 1 + Fold 3
  예측: Fold 2

3차
  학습: Fold 1 + Fold 2
  예측: Fold 3

세 번의 부분 예측을 원래 순서로 합치면
  y_pred_rf = [Fold 1 예측, Fold 2 예측, Fold 3 예측을 원래 자리로 합친 결과]
```

- 중요: 예측 결과가 3세트 반환되는 것이 아니라, 모든 샘플에 대한 최종 예측 1세트가 반환됩니다.
- 즉, `y_pred_rf`의 길이는 `y_train`과 같습니다.

In [ ]:
# 📌 §300 RF 학습
# 코드를 작성해주세요 👇

# 예상 출력: (시간이 걸립니다 — 1~3분 소요)

In [ ]:
# 📌 §300 RF 정확도
# 코드를 작성해주세요 👇

# 예상 출력: RF 교차검증 정확도: 0.7395

## 📈 LogisticRegression 학습

S-curve로 확률을 계산하는 로지스틱 회귀로 같은 데이터를 분류합니다. RF보다 빠르고, 텍스트 분류에서 강력한 baseline입니다.

[→ §300 강의노트](../notes/UD-06-300__rf-vs-logreg.md)

[🔗 LogisticRegression 공식 문서](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

In [ ]:
# 📌 §300 LogReg 학습
# 코드를 작성해주세요 👇

# 예상 출력: (RF보다 훨씬 빠릅니다 — 수 초 소요)

In [ ]:
# 📌 §300 LogReg 정확도
# 코드를 작성해주세요 👇

# 예상 출력: LogReg 교차검증 정확도: 0.7871

## 📋 성적표 (classification_report)

전체 정확도만으로는 부족합니다. 카테고리별 Precision, Recall, F1-score를 확인해서 어디를 잘 맞추고 어디를 못 맞추는지 살펴봅니다.

[→ §400 강의노트](../notes/UD-06-400__demo-classification-report.md)

[🔗 classification_report 공식 문서](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)

In [ ]:
# 📌 §400 LogReg 성적표
# 코드를 작성해주세요 👇

# 예상 출력: 스포츠 F1≈0.92 (최고), 사회 F1≈0.65 (최저)

In [ ]:
# 📌 §400 RF 성적표
# 코드를 작성해주세요 👇

# 예상 출력: RF도 스포츠 최고, 사회 최저 — LogReg보다 전반적으로 살짝 낮음

## 🔥 혼동행렬

어떤 카테고리끼리 혼동하는지 7×7 행렬로 확인합니다. 대각선이 맞춘 건수, 나머지가 틀린 건수입니다.

[→ §400 강의노트](../notes/UD-06-400__demo-classification-report.md)

In [ ]:
# 📌 §400 혼동행렬 계산
# 코드를 작성해주세요 👇

# 예상 출력: 7×7 혼동행렬 DataFrame

In [ ]:
# 📌 §400 혼동행렬 시각화
# 코드를 작성해주세요 👇

# 예상 출력: 대각선이 진하고, 사회-정치 영역이 밝은 heatmap

In [ ]:
# 📌 §400 혼동 패턴
# 코드를 작성해주세요 👇

# 예상 출력: 사회→정치 456건, 정치→사회 618건, 경제→사회 789건

## 🔍 predict_proba — 모델의 확신도

predict()는 카테고리 번호만 알려주지만, predict_proba()는 각 카테고리에 대한 확률을 보여줍니다. 모델이 얼마나 확신하는지 들여다봅니다.

[→ §300 강의노트](../notes/UD-06-300__rf-vs-logreg.md)

In [ ]:
# 📌 §300 LogReg 확률 예측
# 코드를 작성해주세요 👇

# 예상 출력: 3행 × 7열 — 각 행의 합이 1.0, 가장 높은 확률이 예측 카테고리

In [ ]:
# 📌 §300 RF 확률 예측
# 코드를 작성해주세요 👇

# 예상 출력: RF는 투표 비율이라 확률이 더 뭉뚝한 형태 (0.7~0.9 vs 0.01~0.05)

## ⚖️ 모델 비교 정리

같은 TF-IDF 벡터에서 분류기만 바꿨을 때, 어떤 차이가 나는지 정리합니다.

[→ §400 강의노트](../notes/UD-06-400__demo-classification-report.md)

In [ ]:
# 📌 §400 모델 비교
# 코드를 작성해주세요 👇

# 예상 출력: RF 0.7395, LogReg 0.7871

## 📊 모델이 어떤 단어를 증거로 썼는가

RF가 뉴스를 분류할 때 어떤 단어에 주목했는지 `feature_importances_`로 확인합니다.

[→ §500 강의노트](../notes/UD-06-500__eda-feature-importance.md)

[🔗 RandomForestClassifier.feature_importances_ 공식 문서](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.feature_importances_)

In [ ]:
# 📌 §500 RF feature_importances_ Top 20
feat_names = tfidf.get_feature_names_out()
importances = rf.feature_importances_

fi_df = pd.DataFrame({"단어": feat_names, "중요도": importances})
fi_df = fi_df.sort_values("중요도", ascending=False).head(20)
fi_df = fi_df.sort_values("중요도")  # barh: 오름차순 → 위에 높은 값

plt.figure(figsize=(8, 7))
sns.barplot(data=fi_df, x="중요도", y="단어", color="steelblue")
plt.title("RF Feature Importances — Top 20")
plt.xlabel("중요도 (평균 불순도 감소량)")
plt.tight_layout()
plt.show()
# 예상 출력: 수평 막대그래프 — 스포츠 관련 단어가 상위 독점

In [ ]:
# 📌 §500 상위 단어 출력 및 해석
# ✍️ hint: fi_df에서 상위 10개를 출력하고, 스포츠/사회 단어 패턴을 해석해보세요
# 코드를 작성해주세요 👇

# 예상 출력: Top 10 단어 테이블 + 해석 메시지

## 🔤 도전: 형태소 분석으로 성능 올리기

현재 전처리는 숫자 제거 + 소문자 변환뿐입니다. Pecab 형태소 분석기로 조사와 구두점을 제거하면 성능이 달라질까요?

> ⏱️ **실행 시간 안내**: Pecab 전처리는 45,654건 기준 **3~5분** 소요됩니다.

[→ §500 강의노트](../notes/UD-06-500__eda-feature-importance.md)

In [ ]:
# 📌 §600 Pecab 설치 및 임포트
# ✍️ hint: !uv pip install pecab -q 후 PeCab을 임포트하세요
# 코드를 작성해주세요 👇

# 예상 출력: Pecab 초기화 완료

In [ ]:
# 📌 §600 Pecab 전처리 함수
# ✍️ hint: pecab.pos(text, drop_space=False)로 형태소 분석 후, 조사(J*)와 문장부호(SF) 제거
# 코드를 작성해주세요 👇

# 예상 출력: 원문 → Pecab 전처리 결과 비교

In [ ]:
# 📌 §600 Pecab 전처리 적용
# ✍️ hint: train2 = pd.read_csv("data/klue/train_data.csv") 원본 재로드 후 apply()
# 코드를 작성해주세요 👇

# 예상 출력: 전처리 완료: 45654건 + 원문/Pecab 비교 3건

In [ ]:
# 📌 §600 Pecab TF-IDF + LogReg
# ✍️ hint: 기존과 동일한 TfidfVectorizer + LogisticRegression + cross_val_predict
# 코드를 작성해주세요 👇

# 예상 출력: Pecab LogReg 정확도: ~0.82

In [ ]:
# 📌 §600 전처리 방법 비교
# ✍️ hint: lr_accuracy (기존)와 pc_accuracy (Pecab)를 비교 출력
# 코드를 작성해주세요 👇

# 예상 출력: 두 방법 정확도 비교 + 차이